# Пример 05. Множество решений неоднородной системы Ax=b

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** неоднородные линейные системы, общее решение, частное решение и базис нуль-пространства; связь с линейной и гребневой регрессией.

## Условие

Для заданных матрицы $A$ и вектора $b$ требуется описать множество всех решений системы $Ax = b$. Структура решения должна быть представлена в виде суммы частного решения $x_p$ и линейной комбинации базисных векторов нуль-пространства $\ker(A)$.

## Математическая идея

Множество решений неоднородной системы $Ax = b$ имеет вид

$$\{x \mid Ax = b\} = x_p + \ker(A) = \{x_p + v \mid v\in\ker(A)\},$$

где $x_p$ — любое частное решение, а $\ker(A) = \{v \mid Av = 0\}$ — нуль-пространство (ядро) матрицы $A$.

Если $\ker(A) = \mathrm{span}\{u_1, \dots, u_k\}$, то общее решение:

$$x = x_p + \sum_{i=1}^{k} \alpha_i u_i, \quad \alpha_i\in\mathbb{R}.$$

Размерность $\ker(A)$ равна $n - \mathrm{rank}(A)$ для матрицы $A\in\mathbb{R}^{m\times n}$.

## Решение

1. По матрице $A\in\mathbb{R}^{4\times 5}$ и вектору $b\in\mathbb{R}^4$ выписывается частное решение $x_p$.
2. Находятся два линейно независимых вектора нуль-пространства $u_1, u_2$.
3. Общее решение записывается как $x = x_p + \alpha_1 u_1 + \alpha_2 u_2$.

## Реализация на Python

Вектор `x_particular` и список `nullspace_basis` задаются напрямую на основе аналитического решения. В проверке используется матричное умножение `A @ x`, чтобы убедиться, что для конкретных значений параметров выполнено равенство $Ax = b$. Во второй части Notebook'а сравниваются обычная и гребневая регрессии на вырожденных данных: третий признак линейно зависит от первых двух, что делает матрицу $X^\top X$ плохо обусловленной.


In [3]:
import numpy as np

# Пункт b
A = np.array([[1, -1, 0, 0, 1],
             [1, 1, 0, -3, 0],
             [2, -1, 0, 1, -1],
             [-1, 2, 0, -2, -1]])

b = np.array([3, 6, 5, -1])

# Найдем частное решение и базис нулевого пространства
x_particular = np.array([3, 0, 0, -1, 0])
nullspace_basis = [
    np.array([1, 2, 0, 1, 1]),
    np.array([0, 0, 1, 0, 0])
]

print("Частное решение:", x_particular)
print("Базис нулевого пространства:", nullspace_basis)

Частное решение: [ 3  0  0 -1  0]
Базис нулевого пространства: [array([1, 2, 0, 1, 1]), array([0, 0, 1, 0, 0])]


## Дополнительный пример

**Идея.** В линейной регрессии веса $w$ находятся из системы $(X^\top X) w = X^\top y$. Если столбцы $X$ линейно зависимы, то $X^\top X$ вырождена, и решение либо не существует, либо не единственно (существует нуль-пространство). Гребневая регрессия добавляет к матрице $X^\top X$ слагаемое $\lambda I$, что делает систему однозначно разрешимой.

**Что демонстрирует код.** Признак $x_3 = x_1 + x_2$ делает матрицу признаков вырожденной. `LinearRegression` всё же находит решение, но веса получаются нестабильными. `Ridge(alpha=1.0)` добавляет $L_2$-регуляризацию и даёт устойчивые, интерпретируемые веса.


### Регрессия с регуляризацией vs без

In [1]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression

# Создадим вырожденные данные (линейно зависимые признаки)
np.random.seed(42)
X = np.random.randn(100, 3)
X[:, 2] = X[:, 0] + X[:, 1]  # Третий признак = сумма первых двух
y = X[:, 0] + 2 * X[:, 1] + np.random.randn(100) * 0.1

# Обычная линейная регрессия
lr = LinearRegression()
lr.fit(X, y)
print("Обычная регрессия (веса):", lr.coef_)

# Гребневая регрессия (λ=1.0)
ridge = Ridge(alpha=1.0)
ridge.fit(X, y)
print("Гребнева регрессия (веса):", ridge.coef_)

Обычная регрессия (веса): [-0.00281239  0.9997026   0.99689021]
Гребнева регрессия (веса): [2.76937959e-04 9.92914353e-01 9.93191291e-01]


## Проверка результата

Для основного решения проверка состоит в подстановке конкретного набора параметров $\alpha_i$ в формулу общего решения и проверке равенства $Ax = b$ через `A @ x`. Для дополнительного примера сравниваются веса обычной и гребневой регрессии: у гребневой регрессии веса имеют меньшую норму и более устойчивы.

## Вывод

Структура множества решений линейной системы — частное решение плюс нуль-пространство — напрямую объясняет, почему регрессия на линейно зависимых признаках оказывается неустойчивой. Регуляризация (в данном случае $L_2$) эффективно «сужает» нуль-пространство и делает решение единственным и устойчивым.
